# 02 — Data Cleaning & Table Integration

This notebook reads the original CSVs, fixes basic data types, aggregates item/payment rows to order level, joins the related tables, and creates the first clean order-level dataset.

The output is `data/processed/orders_clean.csv`.

**Target definition:** an order is late when its actual customer delivery timestamp is later than its estimated delivery date. Only delivered orders with both dates are retained because their target is observable.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
RAW, PROCESSED = ROOT / "data" / "raw", ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

def load_csv(filename):
    path = RAW / filename
    if not path.exists():
        raise FileNotFoundError(f"Please put {filename} inside {RAW}")
    return pd.read_csv(path)

orders = load_csv("olist_orders_dataset.csv")
items = load_csv("olist_order_items_dataset.csv")
customers = load_csv("olist_customers_dataset.csv")
sellers = load_csv("olist_sellers_dataset.csv")
products = load_csv("olist_products_dataset.csv")
payments = load_csv("olist_order_payments_dataset.csv")
geo = load_csv("olist_geolocation_dataset.csv")

## 1. Fix date columns and inspect duplicates

In [2]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

print("Duplicate order IDs:", orders["order_id"].duplicated().sum())
print("Duplicate customer IDs:", customers["customer_id"].duplicated().sum())
print("Duplicate seller IDs:", sellers["seller_id"].duplicated().sum())

orders = orders.drop_duplicates(subset="order_id")
customers = customers.drop_duplicates(subset="customer_id")
sellers = sellers.drop_duplicates(subset="seller_id")
products = products.drop_duplicates(subset="product_id")

Duplicate order IDs: 0
Duplicate customer IDs: 0
Duplicate seller IDs: 0


## 2. Aggregate item and payment tables

In [3]:
# One row per order. An order may contain several purchased items.
item_order = items.groupby("order_id").agg(
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    item_count=("order_item_id", "count"),
    seller_id=("seller_id", "first"),
    product_id=("product_id", "first"),
).reset_index()

payment_order = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_type=("payment_type", "first"),
).reset_index()

print("Item aggregation:", item_order.shape)
print("Payment aggregation:", payment_order.shape)
display(item_order.head())

Item aggregation: (98666, 6)
Payment aggregation: (99440, 4)


,order_id,total_price,total_freight,item_count,seller_id,product_id
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,9d7a1d34a5052409006425275ba1c2b4,7634da152a4610f1595efa32f14722fc
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,df560393f3a51e74553ab94004ba5c87,ac6c3623068f30de03045865e4e10089


## 3. Prepare approximate location coordinates

In [4]:
geo["geolocation_zip_code_prefix"] = (
    geo["geolocation_zip_code_prefix"].astype(str).str.zfill(5)
)
geo_centroids = geo.groupby("geolocation_zip_code_prefix").agg(
    latitude=("geolocation_lat", "mean"),
    longitude=("geolocation_lng", "mean"),
).reset_index()

customers["customer_zip_code_prefix"] = (
    customers["customer_zip_code_prefix"].astype(str).str.zfill(5)
)
sellers["seller_zip_code_prefix"] = (
    sellers["seller_zip_code_prefix"].astype(str).str.zfill(5)
)

customer_geo = customers.merge(
    geo_centroids,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
).rename(columns={"latitude": "customer_lat", "longitude": "customer_lng"})

seller_geo = sellers.merge(
    geo_centroids,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
).rename(columns={"latitude": "seller_lat", "longitude": "seller_lng"})

## 4. Join tables

In [5]:
data = (orders
    .merge(item_order, on="order_id", how="inner")
    .merge(payment_order, on="order_id", how="left")
    .merge(customer_geo[["customer_id", "customer_state", "customer_lat", "customer_lng"]],
           on="customer_id", how="left")
    .merge(seller_geo[["seller_id", "seller_state", "seller_lat", "seller_lng"]],
           on="seller_id", how="left")
    .merge(products[["product_id", "product_category_name"]],
           on="product_id", how="left"))

print("Joined shape:", data.shape)
display(data.head())

Joined shape: (98666, 23)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_price,total_freight,...,payment_value,payment_installments,payment_type,customer_state,customer_lat,customer_lng,seller_state,seller_lat,seller_lng,product_category_name
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,29.99,8.72,...,38.71,1.0,credit_card,SP,-23.576983,-46.587161,SP,-23.680729,-46.444238,utilidades_domesticas
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,118.70,22.76,...,141.46,1.0,boleto,BA,-12.177924,-44.660711,SP,-19.807681,-43.980427,perfumaria
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,159.90,19.22,...,179.12,3.0,credit_card,GO,-16.745150,-48.514783,SP,-21.363502,-48.229601,automotivo
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,45.00,27.20,...,72.20,1.0,credit_card,RN,-5.774190,-35.271143,MG,-19.837682,-43.924053,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,19.90,8.72,...,28.62,1.0,credit_card,SP,-23.676370,-46.514627,SP,-23.543395,-46.262086,papelaria


## 5. Create the target and keep eligible orders

In [6]:
eligible = (
    data["order_status"].eq("delivered")
    & data["order_delivered_customer_date"].notna()
    & data["order_estimated_delivery_date"].notna()
    & data["order_purchase_timestamp"].notna()
)
data = data.loc[eligible].copy()

data["late"] = (
    data["order_delivered_customer_date"]
    > data["order_estimated_delivery_date"]
).astype("int8")

print("Eligible delivered orders:", f"{len(data):,}")
print("Late-order percentage:", f"{data['late'].mean():.2%}")
display(data["late"].value_counts().rename(index={0: "Not late", 1: "Late"}))

Eligible delivered orders: 96,470
Late-order percentage: 8.11%


late
Not late    88644
Late         7826
Name: count, dtype: int64

## 6. Save clean data

In [7]:
output = PROCESSED / "orders_clean.csv"
data.to_csv(output, index=False)
print("Saved:", output)
print("Final shape:", data.shape)

Saved: c:\Users\Administrator\Desktop\olist_notebooks\data\processed\orders_clean.csv
Final shape: (96470, 24)


## Cleaning notes

- Cancelled and unavailable orders are excluded because this target is based on an observed delivery date.
- Multiple item rows and payment rows are aggregated to one order row.
- Missing values are not blindly filled here; model preprocessing will handle them inside the training pipeline.
- Geolocation coordinates are approximate postal-code centroids.
- This dataset is historical Brazilian marketplace data, not live Noon/Jumia data.